# 07 · Clustering con DBSCAN

> **Objetivo:** aplicar DBSCAN, entender sus parámetros, y comparar
> sus resultados con K-Means.

## ¿Qué vamos a hacer?

1. Repasar el algoritmo y sus dos hiperparámetros.
2. Elegir `eps` con la curva k-distance.
3. Probar diferentes configuraciones.
4. Visualizar los clusters y los outliers.

## Concepto teórico

DBSCAN (Density-Based Spatial Clustering of Applications with Noise)
agrupa puntos basándose en **densidad local**, no en distancia a un centroide.

### Definiciones clave

- **eps (ε)**: radio del vecindario.
- **min_samples**: nº mínimo de puntos en el vecindario para considerar
  un punto como **núcleo**.
- **Punto núcleo**: tiene ≥ `min_samples` vecinos dentro de ε.
- **Punto frontera**: está dentro del vecindario de un núcleo, pero
  no tiene él mismo ≥ `min_samples` vecinos.
- **Ruido (noise)**: ni núcleo ni frontera. Etiqueta `-1`.

### Diferencias clave con K-Means

| K-Means | DBSCAN |
|---|---|
| Necesita `k` | No, descubre el nº de clusters. |
| Asume clusters esféricos | Encuentra **formas arbitrarias**. |
| Asigna TODO punto | Marca outliers como `-1`. |
| Sensible a outliers | Robusto (los aísla). |
| Escala bien a alta dimensión | Sufre con muchas dimensiones (curse of dimensionality). |

> **Tip:** DBSCAN brilla cuando esperas que existan *outliers genuinos*
> (mayoristas, fraudes) que no quieres forzar a un cluster.


In [ ]:
# Permite importar el paquete src/ desde el notebook
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA

from src.config import (
    FEATURES_DATA_FILE,
    EXTENDED_NUMERIC_FEATURES,
    DBSCAN_MODEL_FILE,
    PIPELINE_FILE,
    RANDOM_STATE,
)
from src.features.preprocessing import build_preprocessing_pipeline
from src.models.clustering import evaluate_clustering, fit_dbscan
from src.visualization.plots import plot_clusters_2d, plot_k_distance


## 1. Preparar features

In [ ]:
features = pd.read_parquet(FEATURES_DATA_FILE)
pipeline = build_preprocessing_pipeline(
    numeric_features=EXTENDED_NUMERIC_FEATURES,
    use_log=True,
)
X = pipeline.fit_transform(features[EXTENDED_NUMERIC_FEATURES])
print(f"Matriz: {X.shape}")


## 2. Elegir `eps` con la curva k-distance

Heurística clásica (Ester et al., 1996):

1. Calcula la distancia de cada punto a su `min_samples`-ésimo vecino.
2. Ordena esas distancias de menor a mayor.
3. Grafica → busca el "codo".
4. El valor del codo en el eje y es un buen `eps` inicial.


In [ ]:
MIN_SAMPLES = 2 * X.shape[1]  # heurística: 2 * nº de features
print(f"min_samples sugerido: {MIN_SAMPLES}")

fig = plot_k_distance(X, k=MIN_SAMPLES)
plt.show()


> **Tip:** el codo está donde la curva pasa de plana a empinada.
> Para este dataset suele estar en `eps ≈ 0.5 - 1.0`. Vamos a probar varios.

## 3. Comparar varias combinaciones de hiperparámetros


In [ ]:
configs = [
    (0.3, MIN_SAMPLES),
    (0.5, MIN_SAMPLES),
    (0.7, MIN_SAMPLES),
    (1.0, MIN_SAMPLES),
    (0.5, MIN_SAMPLES // 2),
    (0.5, MIN_SAMPLES * 2),
]

rows = []
for eps, ms in configs:
    model = fit_dbscan(X, eps=eps, min_samples=ms)
    metrics = evaluate_clustering(X, model.labels_)
    rows.append({"eps": eps, "min_samples": ms, **metrics})

results = pd.DataFrame(rows)
results


**Cómo interpretar la tabla:**

- `n_clusters` = clusters reales (sin contar el -1 de ruido).
- `n_noise` = puntos clasificados como outliers.
- Una buena configuración:
  - tiene `n_clusters >= 2` (si es 1, eps está demasiado grande).
  - tiene `n_noise` razonable (~5-15%; si es 50%+, eps está demasiado
    pequeño).
  - tiene silueta positiva.

> **Errores comunes:**
>
> - **eps muy pequeño** → casi todo es ruido.
> - **eps muy grande** → un solo cluster gigante.
> - **min_samples muy alto** → muchos puntos quedan sin clasificar.

## 4. Configuración final


In [ ]:
EPS_FINAL = 0.5
MS_FINAL = MIN_SAMPLES
dbscan = fit_dbscan(X, eps=EPS_FINAL, min_samples=MS_FINAL)
features["cluster_dbscan"] = dbscan.labels_

print("Distribución de clientes (incluye -1 = ruido):")
print(features["cluster_dbscan"].value_counts().sort_index())


## 5. Visualización 2D

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_2d = pca.fit_transform(X)

fig = plot_clusters_2d(
    X_2d,
    dbscan.labels_,
    title=f"DBSCAN · eps={EPS_FINAL}, min_samples={MS_FINAL}",
)
plt.show()


> Los puntos grises son ruido. Observa cómo DBSCAN los separa
> automáticamente, mientras que K-Means los habría asignado a algún
> cluster jalando los centroides.

## 6. ¿Quiénes son los outliers?


In [ ]:
outliers = features[features["cluster_dbscan"] == -1]
print(f"Outliers detectados: {len(outliers)}")
print("\nTop 10 outliers por Monetary:")
outliers.nlargest(10, "Monetary")


**Interpretación de negocio:**

Estos suelen ser:
- **Mayoristas** con cantidades enormes.
- **Cuentas corporativas** con compras esporádicas pero altas.
- **Casos atípicos** que merecen atención manual del equipo de ventas.

DBSCAN te los entrega "gratis", lo cual es invaluable.

## 7. Limitaciones de DBSCAN

- En **alta dimensión** (>10), la noción de "densidad" se debilita
  (curse of dimensionality). Considera reducir con PCA antes.
- Si los clusters tienen densidades muy diferentes, un único `eps` no
  funciona bien. Considera **HDBSCAN** (versión jerárquica).
- No siempre encuentra el número de clusters que el negocio necesita.

## 8. Guardar el modelo


In [ ]:
joblib.dump(dbscan, DBSCAN_MODEL_FILE)
print(f"DBSCAN guardado en: {DBSCAN_MODEL_FILE}")


## Resumen

| Aspecto | DBSCAN |
|---|---|
| Entrada principal | `eps`, `min_samples`. |
| Cómo elegir `eps` | Curva k-distance. |
| Salida especial | Etiqueta `-1` para outliers. |
| Forma de clusters | Arbitraria (no esférica). |
| Cuándo usarlo | Cuando esperas outliers genuinos o clusters no convexos. |

---

## Preguntas de Reflexión

1. ¿Por qué `min_samples` ≥ 2 × nº features es una buena heurística?
2. ¿Qué pasaría si aplicaras DBSCAN sin escalar las features?
3. Si DBSCAN encuentra 12 clusters muy pequeños, ¿qué le aconsejarías
   al negocio?
4. ¿En qué situaciones K-Means es preferible a DBSCAN, y viceversa?

> **Próximo paso:** ``08_model_comparison_and_interpretation.ipynb`` —
> comparar ambos modelos y traducir los clusters en perfiles accionables.
